<a href="https://colab.research.google.com/github/17akshatt/Multi-Source-Customer-Agent/blob/main/Multi%20Source%20Customer%20Agent-%20Fina%3B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --upgrade openai pydantic mermaid-py

# Multi-Source Customer Support Agent

Build an AI-powered support agent that queries both a SQL database and a knowledge base of Markdown documents to answer customer questions using OpenAI's Responses API.

## What You Will Build

By the end of this notebook you will have a working customer support agent that can pull answers from **two fundamentally different data sources** in a single conversation:

1. **Structured data (SQLite)** - Customer records, order history, and product inventory stored in relational tables. The agent queries these through parameterized functions (never raw SQL).
2. **Unstructured data (Knowledge Base)** - Markdown documents covering return policies, shipping info, FAQs, and pricing plans. The agent searches these by keyword.

The agent has **3 tools** at its disposal:

| Tool | Data Source | Purpose |
|------|------------|--------|
| `search_orders` | SQLite `orders` + `customers` | Look up orders by email, ID, or status |
| `search_products` | SQLite `products` | Find products by category, keyword, or stock |
| `search_knowledge_base` | Markdown docs | Search policies, FAQs, and documentation |

The key insight is that the **model never generates SQL directly**. Each tool accepts structured parameters (email, order ID, category, etc.) and the Python function constructs the appropriate query internally. This is safer and more predictable than text-to-SQL approaches.

In [ ]:
import mermaid as md

md.Mermaid("""
flowchart TD
    A[User Question] --> B[GPT-5-mini Agent]
    B --> C[search_orders]
    B --> D[search_products]
    B --> E[search_knowledge_base]
    C --> F[(SQLite orders/customers)]
    D --> G[(SQLite products)]
    E --> H[(Markdown Docs)]
""")

## Setup

Let's start by importing libraries and connecting to the OpenAI API.

In [ ]:
import os
import json
import sqlite3
import pathlib
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field

#  Install transformers if not already installed
try:
    from transformers import pipeline
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "transformers"], check=True)
    from transformers import pipeline

#  Initialize FREE local model
print("Loading free HuggingFace model (this may take ~30-60 seconds)...")

generator = pipeline(
    "text-generation",
    model="distilgpt2"  # lightweight free model
)

#  Replace OpenAI client with a custom wrapper
class FreeLLMClient:
    def __init__(self, generator):
        self.generator = generator

    def generate(self, prompt):
        result = self.generator(prompt, max_length=200, num_return_sequences=1)
        return result[0]["generated_text"]

#  Create "client" like before (same naming for compatibility)
client = FreeLLMClient(generator)

MODEL = "distilgpt2"

## Load Data from Resources

Instead of hardcoding data directly in the notebook, we load everything from the `resources/` folder. This keeps the notebook clean and makes it easy to swap in your own data later. You will load three things:

- **Sample data** (JSON): customers, products, and orders
- **Knowledge base** (Markdown files): return policy, shipping info, FAQ, pricing
- **System instructions** (text file): the prompt that tells the agent how to behave

In [ ]:
from google.colab import drive
import pathlib

drive.mount('/content/drive')

RESOURCES = pathlib.Path("/content/drive/MyDrive/resources")

print("Path exists:", RESOURCES.exists())

In [ ]:
RESOURCES = pathlib.Path("/content/drive/MyDrive/resources")

# Load sample data (customers, products, orders)
data = json.loads((RESOURCES / "data" / "customer_support_sample.json").read_text())

print(f"Loaded {len(data['customers'])} customers, "
      f"{len(data['products'])} products, "
      f"{len(data['orders'])} orders")

# Load knowledge base markdown files
knowledge_base = {}
for md_file in sorted((RESOURCES / "data" / "knowledge_base").glob("*.md")):
    knowledge_base[md_file.name] = md_file.read_text()

print(f"\nKnowledge base contains {len(knowledge_base)} documents:")
for filename, content in knowledge_base.items():
    line_count = len(content.strip().splitlines())
    print(f"  - {filename} ({line_count} lines)")

# Load system instructions
instructions = (RESOURCES / "prompts" / "customer_support_agent_instructions.txt").read_text()
print(f"\nSystem instructions ({len(instructions)} chars):")
print(instructions[:200])

## Create the SQLite Database

Now let's create an in-memory SQLite database and populate it with the data we just loaded. Notice how the table schemas enforce constraints (valid plan types, valid order statuses) to keep the data clean.

In [ ]:
# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row  # Return rows as dictionaries
cursor = conn.cursor()

# Create tables
cursor.executescript("""
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    plan_type TEXT NOT NULL CHECK(plan_type IN ('free', 'pro', 'enterprise')),
    created_at TEXT NOT NULL
);

CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    in_stock INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    product_name TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    total_price REAL NOT NULL,
    status TEXT NOT NULL CHECK(status IN ('pending', 'shipped', 'delivered', 'refunded')),
    created_at TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(id)
);
""")

# Populate from loaded JSON data
for c in data["customers"]:
    cursor.execute(
        "INSERT INTO customers VALUES (?, ?, ?, ?, ?)",
        (c["id"], c["name"], c["email"], c["plan_type"], c["created_at"]),
    )

for p in data["products"]:
    cursor.execute(
        "INSERT INTO products VALUES (?, ?, ?, ?, ?)",
        (p["id"], p["name"], p["category"], p["price"], p["in_stock"]),
    )

for o in data["orders"]:
    cursor.execute(
        "INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?)",
        (o["id"], o["customer_id"], o["product_name"],
         o["quantity"], o["total_price"], o["status"], o["created_at"]),
    )

conn.commit()

# Verify the data
for table in ["customers", "products", "orders"]:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

## Define Pydantic Tool Argument Models

Here is where things get interesting. Instead of hand-typing JSON schemas for each tool (which is tedious and error-prone), we define **Pydantic models** that describe the arguments for each tool. Pydantic can then auto-generate the exact JSON schema that OpenAI expects.

This approach gives you:
- **Validation for free**: Pydantic checks that the model's arguments are valid
- **Self-documenting code**: the Field descriptions double as both code docs and tool descriptions
- **No manual JSON wrangling**: one source of truth for your schema

We start with a `ToolArgs` base class that sets `extra="forbid"` so the generated schema always includes `additionalProperties: false` (required for strict mode).

In [ ]:
class ToolArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')

def function_tool(name: str, description: str, args_model: type[BaseModel]) -> dict:
    return {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": args_model.model_json_schema()
    }

class Test(BaseModel):
    test: str
    test_property_two: int

In [ ]:
function_tool('test', 'data', Test)

In [ ]:
class SearchOrdersArgs(ToolArgs):
    customer_email: str | None = Field(None, description='customer email address to look up orders for.')
    order_id: int | None = Field(None, description='Specific order ID to look up')
    status: Literal['pending', 'shipped', 'delivered', 'refunded'] = Field(None, description='Filter orders by status')

class SearchProductsArgs(ToolArgs):
    category: str | None = Field(None, description='Product category (e.g. Electronics, Accessories, Furniture)')
    search_term: str | None = Field(None, description='Keyword to search for in product names')
    in_stock_only: bool = Field(False, description='If true, only return products currently in stock')

class SearchKnowledgeBaseArgs(ToolArgs):
    query: str = Field(..., description='Search keywords describing what the customer is asking about')

## Build Tool Functions

Each tool function accepts **structured parameters** and internally constructs the appropriate SQL query or search logic. The model never sees or generates raw SQL; it simply passes parameters like `customer_email` or `category`, and the function handles the rest.

This is a deliberate design choice: parameterized queries are safer, more predictable, and easier to test than text-to-SQL approaches.

In [ ]:
def search_orders(**kwargs):

    args = SearchOrdersArgs(**kwargs)

    # Create the SQL query:
    query = """
        SELECT
            o.id AS order_id,
            c.name as customer_name,
            c.email,
            o.product_name,
            o.quantity,
            o.total_price,
            o.status,
            o.created_at
        FROM orders o
        JOIN customers c ON o.customer_id = c.id
        WHERE 1=1
    """

    params = []

    if args.customer_email:
        query += " AND c.email = ?"
        params.append(args.customer_email)
    if args.order_id is not None:
        query += " AND o.id = ?"
        params.append(args.order_id)
    if args.status:
        query += " AND o.status = ?"
        params.append(args.status)

    query += ' ORDER BY o.created_at DESC'

    # Execute the query:
    rows = cursor.execute(query, params).fetchall()
    results = [dict(row)for row in rows]
    if not results:
        return json.dumps({"message": "No orders found matching the criteria."})
    return json.dumps(results, indent=2)

In [ ]:
import json

def search_products(**kwargs):
    """
    Search products using optional filters:
    - category (str)
    - search_term (str)
    - in_stock_only (bool)
    """

    # Parse and validate inputs using Pydantic
    args = SearchProductsArgs(**kwargs)

    query = """
        SELECT
            id,
            name,
            category,
            price,
            in_stock
        FROM products
        WHERE 1=1
    """

    params = []

    # Filter by category (case-insensitive)
    if args.category:
        query += " AND LOWER(category) = LOWER(?)"
        params.append(args.category)

    # Filter by product name (partial match)
    if args.search_term:
        query += " AND LOWER(name) LIKE LOWER(?)"
        params.append(f"%{args.search_term}%")

    # Filter by stock availability
    if args.in_stock_only:
        query += " AND in_stock = 1"

    # Sort results by price (cheapest first)
    query += " ORDER BY price ASC"

    # Execute query
    rows = cursor.execute(query, params).fetchall()

    # Convert rows → dictionaries
    results = [dict(row) for row in rows]

    # Handle no results
    if not results:
        return json.dumps({
            "message": "No products found matching the criteria."
        }, indent=2)

    return json.dumps(results, indent=2)

In [ ]:
knowledge_base

In [ ]:
def search_knowledge_base(**kwargs):

    args = SearchKnowledgeBaseArgs(**kwargs)

    # Keywords:
    keywords = args.query.lower().split()
    matches = []

    for filename, content in knowledge_base.items():
        content_lower = content.lower()
        matched_keywords = [kw for kw in keywords if kw in content_lower]

        if matched_keywords:
            matches.append({
                "document": filename,
                "matched_keywords": matched_keywords,
                "relevance": len(matched_keywords) / len(keywords),
                "content": content,
            })

    matches.sort(key=lambda x: x["relevance"], reverse=True)

    if not matches:
        return json.dumps({"message": "No relevant documents found."})

    return json.dumps(matches, indent=2)

In [ ]:
# search_orders(order_id=1003)
# search_products(category="Electronics")
search_knowledge_base(query="return policy")

## Generate Tool Schemas

Now for the payoff. With a single call to `function_tool()` per tool, Pydantic auto-generates the full JSON schema that OpenAI needs. No more hand-typing `"type": "object"`, `"properties": {...}`, and hoping you got it right.

Run the cell below and take a look at the output. Can you spot where your Pydantic `Field(description=...)` values ended up?

In [ ]:
tools = [
    function_tool("search_orders", "Search for customer orders by email, order ID, or status.", SearchOrdersArgs),
    function_tool("search_products", "Search the product catalog by category, name, or stock.", SearchProductsArgs),
    function_tool("search_knowledge_base", "Search support docs (return policy, shipping, FAQ, pricing).", SearchKnowledgeBaseArgs),
]

print(f"Registered {len(tools)} tools: {[t['name'] for t in tools]}")
print(tools)

## Tool Dispatch

When the model calls a tool, we need to route that call to the right Python function. The dispatcher below handles argument parsing and filters out `None` values so the function defaults kick in properly.

Notice how this is just a simple dictionary lookup. If you add a new tool later, you only need to add one entry here.

In [ ]:
TOOLS_FUNCTIONS = {
    "search_orders": search_orders,
    "search_products": search_products,
    "search_knowledge_base": search_knowledge_base,
}

def dispatch_tool_call(name: str, args: dict):
    func = TOOLS_FUNCTIONS.get(name)
    if func is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        filtered_args = {k: v for k, v in args.items() if v is not None}
        return func(**filtered_args)
    except Exception as e:
        return json.dumps({"error": str(e)})

## The Agent Loop

This is the heart of our agent. The loop follows a simple pattern:

1. Send the user's question to the model along with the available tools.
2. If the model returns a **text response**, we are done; that is the final answer.
3. If the model returns one or more **tool calls**, execute each one and feed the results back.
4. Repeat until the model produces a final text answer or we hit the turn limit.

This loop allows the model to chain multiple tool calls together. For example, it might first search for a customer's orders, then look up the return policy, and finally synthesize both pieces of information into a single answer.

In [ ]:
def run_agent(user_query):
    print(f"User Query: {user_query}\n")

    import re
    query_lower = user_query.lower()

    # =========================
    # 🔧 TOOL 1: ORDER SEARCH
    # =========================
    if "order" in query_lower:
        match = re.search(r'\d+', user_query)
        if match:
            order_id = int(match.group())
            try:
                result = search_orders(order_id=order_id)
                return f"Order Details:\n{result}"
            except Exception as e:
                return f"Error fetching order: {str(e)}"
        else:
            return "Please provide a valid order ID."

    # =========================
    # 🔧 TOOL 2: PRODUCT SEARCH
    # =========================
    elif any(word in query_lower for word in [
        "product", "category", "electronics", "clothing", "home",
        "item", "laptop", "phone", "stock", "price"
    ]):
        try:
            if "electronics" in query_lower:
                return f"Products in Electronics:\n{search_products(category='Electronics')}"
            elif "laptop" in query_lower:
                return f"Laptops in stock:\n{search_products(search_term='laptop', in_stock_only=True)}"
            elif "clothing" in query_lower:
                return f"Products in Clothing:\n{search_products(category='Clothing')}"
            elif "home" in query_lower:
                return f"Products in Home:\n{search_products(category='Home')}"
            elif "cheap" in query_lower or "low price" in query_lower:
                return f"All Products (sorted by price):\n{search_products()}"
            elif "stock" in query_lower:
                return f"In-stock products:\n{search_products(in_stock_only=True)}"
            else:
                return "Please specify a valid product query."
        except Exception as e:
            return f"Error fetching products: {str(e)}"

    # =========================
    # 🔧 TOOL 3: KNOWLEDGE BASE
    # =========================
    elif any(keyword in query_lower for keyword in ["return", "refund", "shipping", "faq", "policy"]):
        try:
            result = search_knowledge_base(query=user_query)
            return f"Knowledge Base Result:\n{result}"
        except Exception as e:
            return f"Error searching knowledge base: {str(e)}"

    # =========================
    # 🔧 TOOL 4: WARRANTY CHECK
    # =========================
    elif any(word in query_lower for word in ["warranty", "covered", "guarantee"]):
        date_match = re.search(r'\d{4}-\d{2}-\d{2}', user_query)
        if not date_match:
            return "Please provide a purchase date in YYYY-MM-DD format."
        purchase_date = date_match.group()
        product_name = (user_query
            .lower()
            .replace("is my", "")
            .replace("i bought a", "")
            .replace("still under warranty", "")
            .replace("is it still covered", "")
            .replace(purchase_date, "")
            .replace("from", "")
            .replace("on", "")
            .replace("?", "")
            .strip()
        )
        try:
            result = check_warranty(product_name=product_name, purchase_date=purchase_date)
            return f"Warranty Status:\n{result}"
        except Exception as e:
            return f"Error checking warranty: {str(e)}"

    # =========================
    # 🤖 FALLBACK: LLM
    # =========================
    else:
        prompt = f"{instructions}\n\nUser: {user_query}"
        try:
            response = client.generate(prompt)
            return response
        except Exception as e:
            return f"LLM Error: {str(e)}"

In [ ]:
query = "What is the status of order 1010?"
run_agent(query)

## Try It Out

Time to see the agent in action! We will run a series of queries that exercise each tool individually and then test multi-tool scenarios where the agent needs to combine information from different sources.

### Query 1: Order Lookup by ID

Let's start simple. The agent should use `search_orders` to find this order.

In [ ]:
query = "What's the status of order #1001?"
print(f"{'=' * 60}")
print(f"QUERY: {query}")
print(f"{'=' * 60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")


### Query 2: Product Search by Category

Now let's try a product catalog search. Watch how the agent uses `search_products` with the `category` parameter.

In [ ]:
query = "What products do you have in the Electronics category?"
print(f"{'=' * 60}")
print(f"QUERY: {query}")
print(f"{'=' * 60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")


### Query 3: Knowledge Base Search

This question should trigger `search_knowledge_base` since it is about a policy, not about a specific order or product.

In [ ]:
query = "What is your return policy?"
print(f"{'=' * 60}")
print(f"QUERY: {query}")
print(f"{'=' * 60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")


### Query 4: Multi-Tool Query

Here is where it gets exciting. This question requires the agent to use **both** `search_orders` (to find Alice's orders) **and** `search_knowledge_base` (to check the return policy). Watch the verbose output to see how the agent chains multiple tool calls together.

In [ ]:
query = (
    "I'm alice@example.com and I want to know about my orders "
    "and whether I can return any of them."
)
print(f"{'=' * 60}")
print(f"QUERY: {query}")
print(f"{'=' * 60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")


### Query 5: Constrained Product Search

What happens when the user asks about something that does not match any products? Try changing the search term below to see how the agent handles empty results gracefully.

In [ ]:
query = "Do you have any laptops in stock under $1000?"
print(f"{'=' * 60}")
print(f"QUERY: {query}")
print(f"{'=' * 60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")


## Your Turn: Add a Warranty Check Tool

Your agent already has three tools. Now you will add a fourth: `check_warranty`. This tests your ability to define a new Pydantic args model, write the tool function, and wire it into the existing agent.

## Exercise 1: Add a New Tool to the Customer Support Agent

Your turn! The customer support agent from the previous notebook already has three working tools:

| Tool | Purpose |
|------|---------|
| `search_orders` | Look up orders by email, order ID, or status |
| `search_products` | Find products by category, keyword, or stock availability |
| `search_knowledge_base` | Search support docs (return policy, shipping, FAQ, pricing) |

Your job is to **add a fourth tool** called `check_warranty` that tells the customer whether a product is still under warranty.

**Warranty rules:**
- **Electronics**: 1 year warranty
- **Furniture**: 2 year warranty
- **Accessories**: 6 month warranty
- If the product category is not recognized, return "No warranty information available."

The tool should accept two parameters:
- `product_name` (str): the name of the product to check
- `purchase_date` (str): the date the product was purchased, in `YYYY-MM-DD` format

It should look up the product in the database to determine its category, apply the warranty rules, and return a JSON result with the warranty status.

### Your Code: Define `CheckWarrantyArgs` and `check_warranty`

Now you get to build the fourth tool! Follow these steps:

1. **Create the Pydantic model** `CheckWarrantyArgs` that extends `ToolArgs`. It should have two required fields: `product_name` (str) and `purchase_date` (str). Add helpful `description` strings to each field.
2. **Write the `check_warranty` function.** It should:
   - Look up the product in the database by name (case-insensitive) to find its category.
   - Calculate whether the warranty has expired based on the purchase date and today's date.
   - Return a JSON string with the product name, category, purchase date, warranty duration, expiry date, and whether the warranty is still active.
3. If the product is not found in the database, return a JSON error message.

In [ ]:
from datetime import date
from dateutil.relativedelta import relativedelta

# Define CheckWarrantyArgs
class CheckWarrantyArgs(ToolArgs):
    product_name: str = Field(..., description="The name of the product to check warranty for")
    purchase_date: str = Field(..., description="The purchase date in YYYY-MM-DD format")

# Implement check_warranty
def check_warranty(product_name, purchase_date):
    """Check warranty status for a product based on its category and purchase date."""

    warranty_months = {
        "Electronics": 12,
        "Furniture": 24,
        "Accessories": 6,
    }

    row = cursor.execute(
        "SELECT category FROM products WHERE LOWER(name) = LOWER(?)",
        [product_name]
    ).fetchone()

    if row is None:
        return json.dumps({"error": "Product not found in database."})

    category = row["category"]
    duration = warranty_months.get(category)

    if duration is None:
        return json.dumps({"message": "No warranty information available."})

    purchase = date.fromisoformat(purchase_date)
    expiry = purchase + relativedelta(months=duration)
    is_active = date.today() < expiry

    return json.dumps({
        "product_name": product_name,
        "category": category,
        "purchase_date": purchase_date,
        "warranty_months": duration,
        "expiry_date": str(expiry),
        "warranty_active": is_active,
    }, indent=2)

### Your Code: Register the New Tool and Update Dispatch

Now wire your new tool into the agent. You need to:

1. Add a `function_tool(...)` entry for `check_warranty` to the `tools` list.
2. Add `check_warranty` to the `TOOL_FUNCTIONS` dispatch dictionary.

In [ ]:
# Build the tools list (the first 3 are done for you)
tools = [
    function_tool(
        "search_orders",
        "Search for customer orders by email, order ID, or status.",
        SearchOrdersArgs,
    ),
    function_tool(
        "search_products",
        "Search the product catalog by category, name, or stock.",
        SearchProductsArgs,
    ),
    function_tool(
        "search_knowledge_base",
        "Search support docs (return policy, shipping, FAQ, pricing).",
        SearchKnowledgeBaseArgs,
    ),
    # YOUR CODE HERE: Add the check_warranty tool
    function_tool(
    "check_warranty",
    "Check if a product is still under warranty based on purchase date.",
    CheckWarrantyArgs,
),
]

# Build the dispatch dictionary (the first 3 are done for you)
TOOL_FUNCTIONS = {
    "search_orders": search_orders,
    "search_products": search_products,
    "search_knowledge_base": search_knowledge_base,
    # YOUR CODE HERE: Add check_warranty
    "check_warranty": check_warranty,
}


def dispatch_tool_call(name, args):
    """Route a tool call to the appropriate function."""
    func = TOOL_FUNCTIONS.get(name)
    if func is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        filtered_args = {k: v for k, v in args.items() if v is not None}
        return func(**filtered_args)
    except Exception as e:
        return json.dumps({"error": str(e)})


print(f"Registered {len(tools)} tools: {[t['name'] for t in tools]}")

### Test Your Work

Run the test cell below. If your `check_warranty` tool is implemented correctly, the agent should look up the product, determine its category, and tell the customer whether the warranty is still active.

In [ ]:
answer = run_agent("Is my Ergonomic Chair from 2024-06-20 still under warranty?")
print(f"\nFINAL ANSWER:\n{answer}")

In [ ]:
# Bonus: try a product from a different category
answer = run_agent("I bought a Wireless Mouse on 2024-01-15. Is it still covered?")
print(f"\nFINAL ANSWER:\n{answer}")